In [7]:
import os
import re
import math
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

In [8]:
nltk.download("vader_lexicon")
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("punkt_tab")

data_path = "kaggle_RC_2019-05.csv"
out_dir = Path("output")
chart_dir = out_dir / "charts"
table_dir = out_dir / "tables"
report_dir = out_dir / "report"

for p in [out_dir, chart_dir, table_dir, report_dir]:
    p.mkdir(parents=True, exist_ok=True)

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [10]:
df = pd.read_csv(data_path)
df.columns = df.columns.str.strip().str.lower()

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)


need = ["body", "subreddit"]
for col in need:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")


Columns: ['subreddit', 'body', 'controversiality', 'score']
Shape: (1000000, 4)


In [11]:
# CLEANING

stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [w for w in word_tokenize(text) if w.isalpha() and w not in stop_words and len(w) > 2]
    return " ".join(tokens)

df["clean"] = df["body"].apply(clean_text)

In [12]:
# remove empty rows
df = df[df["clean"].str.len() > 0].copy()


In [13]:
# Sentiment
sia = SentimentIntensityAnalyzer()
scores = df["clean"].apply(sia.polarity_scores)
df["sentiment"] = scores.apply(lambda x: x["compound"])
df["sent_pos"] = scores.apply(lambda x: x["pos"])
df["sent_neg"] = scores.apply(lambda x: x["neg"])
df["sent_neu"] = scores.apply(lambda x: x["neu"])

def bucket_sentiment(x):
    if x >= 0.05:
        return "positive"
    elif x <= -0.05:
        return "negative"
    return "neutral"

df["sentiment_label"] = df["sentiment"].apply(bucket_sentiment)


In [14]:
# Core metrics
total = len(df)
sub_count = df["subreddit"].nunique()
avg_sent = df["sentiment"].mean()
med_sent = df["sentiment"].median()
pos_rate = (df["sentiment_label"] == "positive").mean()
neg_rate = (df["sentiment_label"] == "negative").mean()
neu_rate = (df["sentiment_label"] == "neutral").mean()

print("\nCore metrics")
print("Total comments:", total)
print("Subreddits:", sub_count)
print("Average sentiment:", round(avg_sent, 4))
print("Median sentiment:", round(med_sent, 4))
print("Positive rate:", round(pos_rate, 4))
print("Negative rate:", round(neg_rate, 4))
print("Neutral rate:", round(neu_rate, 4))



Core metrics
Total comments: 974427
Subreddits: 40
Average sentiment: 0.0877
Median sentiment: 0.0
Positive rate: 0.4503
Negative rate: 0.3001
Neutral rate: 0.2495
